# RAPTOR — Step-by-step verification
Run each section independently. Use `importlib.reload()` to pick up code changes without restarting the kernel.

## 0 - Imports & data
Load one QASPER document. All steps below depend on `document`.

In [ ]:
import os, importlib
from dotenv import load_dotenv
from datasets import load_dataset

load_dotenv()
openai_api_key = "OPEN_API_KEY"

dataset = load_dataset('allenai/qasper', split='train')
document = dataset[21]
print('Title   :', document['title'])
print('Sections:', document['full_text']['section_name'][:5], '...')

Using the latest cached version of the dataset since allenai/qasper couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'qasper' at C:\Users\mickv\.cache\huggingface\datasets\allenai___qasper\qasper\0.3.0\fdc9d8214fbab5dd782958601db4d678e6934a54 (last modified on Thu Apr  2 11:05:27 2026).


Title   : Towards Detection of Subjective Bias using Contextualized Word Embeddings
Sections: ['Introduction', 'Baselines and Approach', 'Baselines and Approach ::: Baselines', 'Baselines and Approach ::: Proposed Approaches', 'Experiments ::: Dataset and Experimental Settings'] ...


## 1 - Chunker
Split the document into ~100-token sentence-preserving chunks.
Paper: Section 3, paragraph 1.

In [3]:
import chunker; importlib.reload(chunker)
from chunker import chunk_qasper_document

chunks = chunk_qasper_document(document, doc_id='qasper_train_0')
print(f'Total chunks : {len(chunks)}')
print(f'Avg tokens   : {sum(c.token_count for c in chunks) / len(chunks):.1f}')
print()
for c in chunks[:3]:
    print(f'  [{c.chunk_index}] section={c.section!r} tokens={c.token_count}')
    print(f'  {c.text[:100]}...')
    print()

Total chunks : 16
Avg tokens   : 75.8

  [0] section='title' tokens=9
  Towards Detection of Subjective Bias using Contextualized Word Embeddings...

  [1] section='abstract' tokens=91
  Subjective bias detection is critical for applications like propaganda detection, content recommenda...

  [2] section='Introduction' tokens=88
  In natural language, subjectivity refers to the aspects of communication used to express opinions, e...



## 2 - Embedder
Embed chunks with SBERT (multi-qa-mpnet-base-cos-v1) into 768-dim L2-normalised vectors.
Paper: Section 3, paragraph 1.

In [4]:
import embedder; importlib.reload(embedder)
from embedder import Embedder, verify_embeddings

embedder_obj = Embedder()
embeddings = embedder_obj.embed_chunks(chunks)
print(f'Embeddings shape: {embeddings.shape}')  # expect (n_chunks, 768)

Loading SBERT model: multi-qa-mpnet-base-cos-v1


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3746.23it/s]
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 768


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.24s/it]

Embeddings shape: (16, 768)


In [5]:
verify_embeddings(embeddings, chunks)


=== Embedding Verification ===

[1] Shape: (16, 768)
    ✓ Shape is correct
[2] ✓ No NaN or Inf values
[3] L2 norms — mean: 1.0000, min: 1.0000, max: 1.0000
    ✓ All vectors are unit-normalized

[4] Cosine similarity spot check:
    Chunk 0 vs Chunk 1 (adjacent):  0.6130
    Chunk 0 vs Chunk -1 (distant):  0.5887

    Top-3 most similar chunks to chunk 0:
    (section: 'title' | 'Towards Detection of Subjective Bias using Contextualized Wo...')
      → Chunk 3 (sim=0.6382) section='Introduction' | 'There has been considerable work on capturing subjectivity u...'
      → Chunk 1 (sim=0.6130) section='abstract' | 'Subjective bias detection is critical for applications like ...'
      → Chunk 4 (sim=0.6127) section='Introduction' | 'However, they primarily focused on detecting and mitigating ...'

=== All checks passed! Ready for clustering. ===



## 3 - Clusterer
Two-pass UMAP + GMM clustering with BIC-optimal cluster count and soft assignment.
Paper: Section 3, Clustering Algorithm.

In [6]:
import clusterer; importlib.reload(clusterer)
from clusterer import cluster_embeddings, verify_clusters

clusters = cluster_embeddings(embeddings)

  Clustering 16 embeddings...
  Pass 1: Global clustering (large n_neighbors)...


c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


    Optimal global clusters (BIC): 3
    Global clusters formed: 3
  Pass 2: Local clustering within each global cluster...
    Local clusters formed: 5
  Total clusters: 8


c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [7]:
verify_clusters(clusters, embeddings, chunks)


=== Cluster Verification ===
Total clusters: 8
  Global: 3  |  Local: 5
  Cluster sizes — min: 2, max: 7, avg: 3.2

  Chunks assigned to >1 cluster (soft): 10 / 16

  Sample clusters:

  Cluster 0 [global] — 3 members
    [5] (Introduction) Their attempt to test their models in a general setting is dwarfed by the fact t...
    [8] (Baselines and Approach ::: Baselines) FastTextBIBREF4: It uses bag of words and bag of n-grams as features for text cl...
    [15] (Conclusion) In this paper, we investigated BERT-based architectures for sentence level subje...

  Cluster 1 [global] — 7 members
    [6] (Introduction) We explore various BERT-based models, including BERT, RoBERTa, ALBERT, with thei...
    [7] (Baselines and Approach) In this section, we outline baseline models like $BERT_{large}$. We further prop...
    [9] (Baselines and Approach ::: Proposed Approaches) Optimized BERT-based models: We use BERT-based models optimized as in BIBREF6 an...

  Cluster 2 [global] — 6 members
    

## 4 - Summarizer
Summarize each cluster with an LLM using the exact Appendix D prompt.
Paper: Section 3, Model-Based Summarization + Appendix D.

In [10]:
import summarizer; importlib.reload(summarizer)
from summarizer import Summarizer, verify_summaries

summarizer_obj = Summarizer(openai_api_key=openai_api_key)
chunk_texts = [c.text for c in chunks]
summary_nodes = summarizer_obj.summarize_clusters(clusters, chunk_texts, level=1)

  Summarizing cluster 0 (3 members, ~269 words)...
  Summarizing cluster 1 (7 members, ~495 words)...
  Summarizing cluster 2 (6 members, ~448 words)...
  Summarizing cluster 4 (2 members, ~175 words)...
  Summarizing cluster 4 (2 members, ~171 words)...
  Summarizing cluster 5 (2 members, ~145 words)...
  Summarizing cluster 6 (2 members, ~123 words)...
  Summarizing cluster 8 (2 members, ~162 words)...


In [11]:
verify_summaries(summary_nodes, chunk_texts)


=== Summarization Verification ===
Total summaries: 8
Avg compression ratio: 0.80  (paper reports ~0.28 — lower is more compressed)

Sample summaries (first 3):

  Cluster 0 | level=1 | sources=[5, 8, 15] | input≈269 words → summary≈188 words
  Summary: This work addresses the challenge of detecting subjective bias in text, specifically focusing on Wikipedia revisions. The authors critique prior attempts that relied on a limited dataset of just 100 i...

  Cluster 1 | level=1 | sources=[6, 7, 9, 10, 12, 13, 14] | input≈495 words → summary≈318 words
  Summary: The study investigates various BERT-based models for subjectivity detection, including BERT, RoBERTa, and ALBERT, in both their base and large configurations, along with their native classifiers. It i...

  Cluster 2 | level=1 | sources=[0, 1, 2, 3, 4, 11] | input≈448 words → summary≈306 words
  Summary: The paper titled "Towards Detection of Subjective Bias using Contextualized Word Embeddings" addresses the challenge of identif

## 5 - Tree
Recursively embed, cluster, and summarize until no further clustering is possible.
Paper: Section 3, Overview of RAPTOR.

In [12]:
import tree; importlib.reload(tree)
from tree import build_tree, verify_tree

raptor_tree = build_tree(chunks, embedder_obj, summarizer_obj)

=== Building RAPTOR Tree ===

[Level 0] Embedding 16 leaf chunks...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.90s/it]
c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Stored 16 leaf nodes.

[Level 0 → 1] Clustering 16 nodes...
  Clustering 16 embeddings...
  Pass 1: Global clustering (large n_neighbors)...
    Optimal global clusters (BIC): 3
    Global clusters formed: 3
  Pass 2: Local clustering within each global cluster...
    Local clusters formed: 5
  Total clusters: 8
  Summarizing 8 clusters with LLM...
  Summarizing cluster 0 (3 members, ~269 words)...


c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Summarizing cluster 1 (7 members, ~495 words)...
  Summarizing cluster 2 (6 members, ~448 words)...
  Summarizing cluster 4 (2 members, ~175 words)...
  Summarizing cluster 4 (2 members, ~171 words)...
  Summarizing cluster 5 (2 members, ~145 words)...
  Summarizing cluster 6 (2 members, ~123 words)...
  Summarizing cluster 8 (2 members, ~162 words)...
  Re-embedding 8 summaries...
  Added 8 summary nodes at level 1.

[Level 1 → 2] Clustering 8 nodes...
  Clustering 8 embeddings...
  Pass 1: Global clustering (large n_neighbors)...
    Optimal global clusters (BIC): 4
    Global clusters formed: 2
  Pass 2: Local clustering within each global cluster...
    Local clusters formed: 2
  Total clusters: 4
  Summarizing 4 clusters with LLM...
  Summarizing cluster 0 (3 members, ~614 words)...


c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Summarizing cluster 2 (3 members, ~645 words)...
  Summarizing cluster 3 (2 members, ~303 words)...
  Summarizing cluster 4 (2 members, ~477 words)...
  Re-embedding 4 summaries...
  Added 4 summary nodes at level 2.

[Level 2 → 3] Clustering 4 nodes...
  Clustering 4 embeddings...
  Pass 1: Global clustering (large n_neighbors)...
    Optimal global clusters (BIC): 3
    Global clusters formed: 1
  Pass 2: Local clustering within each global cluster...
    Local clusters formed: 0
  Total clusters: 1
  Only 1 cluster formed — stopping.

=== Tree complete: 28 total nodes across 3 levels ===



c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [13]:
verify_tree(raptor_tree)


=== Tree Verification ===
Total nodes : 28
Total levels: 3
  Level 0:  16 nodes | avg 76 words/node
  Level 1:   8 nodes | avg 188 words/node
  Level 2:   4 nodes | avg 326 words/node

Sample node at each level:

  Level 0 | node_id=0 | children=[]
  'Towards Detection of Subjective Bias using Contextualized Word Embeddings...'

  Level 1 | node_id=16 | children=[5, 8, 15]
  'This work addresses the challenge of detecting subjective bias in language, specifically focusing on BERT-based models. ...'

  Level 2 | node_id=24 | children=[16, 18, 19]
  'This work tackles the challenge of detecting subjective bias in language, emphasizing the use of BERT-based models. The ...'

=== Tree looks good! Ready for retriever. ===



## 6 - Retriever
Collapsed-tree cosine similarity search with 2000-token budget.
Paper: Section 3, Querying + Appendix F Algorithm 2.

In [14]:
import retriever; importlib.reload(retriever)
from retriever import Retriever, verify_retrieval

retriever_obj = Retriever(raptor_tree, embedder_obj)
result = retriever_obj.retrieve('What discourse relations are used in the proposed method?')

In [15]:
verify_retrieval(result, raptor_tree)


=== Retrieval Verification ===
Query      : 'What discourse relations are used in the proposed method?'
Nodes found: 12 (~1989 words in context)

Nodes by level:
  Level 0: 6 / 16 nodes
  Level 1: 4 / 8 nodes
  Level 2: 2 / 4 nodes

Top-5 retrieved nodes:

  [node_id=22 | level=1 | sim=0.3887]
  'This section discusses baseline models such as $BERT_{large}$ and introduces three advanced approaches for subjectivity ...'

  [node_id=7 | level=0 | sim=0.3718]
  'In this section, we outline baseline models like $BERT_{large}$. We further propose three approaches: optimized BERT-bas...'

  [node_id=10 | level=0 | sim=0.3441]
  'This results in significantly smaller and faster models with performance comparable to their undistilled versions. We fi...'

  [node_id=3 | level=0 | sim=0.3399]
  'There has been considerable work on capturing subjectivity using text-classification models ranging from linguistic-feat...'

  [node_id=5 | level=0 | sim=0.3231]
  'Their attempt to test their models i

## 7 - Pipeline
Summarization subagent. Called by the general agent when a summary request is detected.
Uses root-level nodes as context.

In [17]:
import pipeline; importlib.reload(pipeline)
from pipeline import Pipeline

pipeline_obj = Pipeline(raptor_tree, embedder_obj, openai_api_key=openai_api_key)
pipeline_result = pipeline_obj.run()
print(pipeline_result.answer)

This work addresses the detection of subjective bias in language using advanced BERT-based models. It critiques prior studies that relied on limited, single-article datasets, and instead leverages the extensive WNC corpus—comprising over 423,000 Wikipedia revisions spanning 15 years—to ensure robustness and generalizability. The authors compare traditional models like FastText and BiLSTM with various BERT architectures, including BERT, RoBERTa, and ALBERT, both individually and in ensemble configurations. They employ optimized training strategies, such as larger pretraining datasets, parameter reduction techniques, and knowledge distillation to create efficient models suitable for resource-constrained environments. Experimental results demonstrate that ensemble BERT models significantly outperform baselines, achieving a 5.6-point increase in F1 score and higher accuracy. The study highlights the importance of large, diverse datasets and contextualized embeddings for effective bias dete

## 8 - Evaluator
Score summary quality against the document abstract.
ROUGE-L: lexical overlap. BERTScore: semantic similarity.

In [30]:
import evaluator; importlib.reload(evaluator)
from evaluator import Evaluator

ev = Evaluator(pipeline_obj)
report = ev.evaluate_document(document, doc_id='qasper_train_0')

Summarizing document: Towards Detection of Subjective Bias using Contextualized Word Embeddings
  Computing BERTScore...


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 5985.03it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
report.print()


=== Evaluation Report ===
Documents evaluated : 1
Mean ROUGE-1        : 0.4056
Mean ROUGE-2        : 0.0845
Mean ROUGE-L        : 0.1608
Mean BERTScore F1   : 0.8601

Per-document breakdown:

--------------------------------------------------------------------------------
Document : qasper_train_0
Scores   : ROUGE-1=0.406 | ROUGE-2=0.085 | ROUGE-L=0.161 | BERTScore=0.860
--------------------------------------------------------------------------------
REFERENCE:
Subjective bias detection is critical for applications like propaganda detection, content recommendation, sentiment analysis, and bias neutralization. This bias is introduced in natural language via inflammatory words and phrases, casting doubt over facts, and presupposing the truth. In this work, we perform comprehensive experiments for detecting subjective bias using BERT-based models on the Wiki Neutrality Corpus(WNC). The dataset consists of $360k$ labeled instances, from Wikipedia edits that remove various instances of the